# euroloto — Demo

Toutes les analyses sont dans le package `euroloto`.
Chaque section : **1-5 lignes** de code.


---
## 0. Import & initialisation

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
%matplotlib inline
import euroloto
euroloto.init('../config.yaml')

---
## 0.1 Gestion des données FDJ

| Fonction | Usage | Durée |
|----------|-------|-------|
| `build_tirage()` | **Une seule fois** — crée `tirage.xlsx` | ~30 s |
| `update_tirage()` | **Après chaque tirage** | ~5 s |

In [ ]:
# euroloto.build_tirage('../tirage.xlsx')   # une seule fois
# euroloto.update_tirage('../tirage.xlsx')  # apres chaque tirage
# euroloto.init('../config.yaml')           # recharger apres update
euroloto.info()

---
## 1. Données brutes

In [ ]:
euroloto.show_draws('loto')
euroloto.show_draws('euro')

---
## 2. Fréquences

In [ ]:
euroloto.show_frequency('loto')                # boules principales
euroloto.show_frequency('loto', label='bonus') # numero chance
euroloto.show_frequency('euro')                # EuroMillions

---
## 3. Chaud / Froid

Compare la fréquence des `n_recent` derniers tirages à la moyenne globale.

In [ ]:
euroloto.show_hot_cold('loto', n_recent=50)

---
## 4. Co-occurrences

In [ ]:
euroloto.show_cooccurrence('loto', top_n=20, min_cooc=30)
euroloto.show_cooccurrence('euro', top_n=15)

---
## 5. Tests statistiques & somme

Chi² et KS : p > 0.05 = distribution uniforme.

In [ ]:
euroloto.show_stats('loto')
euroloto.show_stats('euro')

In [ ]:
euroloto.show_sum('loto')

---
## 6. Pair / Impair

In [ ]:
euroloto.show_even_odd('loto')

---
## 7. Retard (numéros en attente)

In [ ]:
euroloto.show_overdue('loto')
euroloto.show_overdue('euro')

---
## 8. Tendance temporelle

In [ ]:
euroloto.show_temporal('euro', top_n=10)
euroloto.show_temporal('loto', top_n=8)

---
## 9. Écarts entre apparitions (gaps)

In [ ]:
euroloto.show_gaps('loto')

---
## 10. Compagnons d'une paire

`lift > 1` = apparaît plus souvent avec la paire qu'attendu.  
`kind='all'` = analyse croisée Loto + EuroMillions.

In [ ]:
FIXED = [26, 27]
euroloto.show_companions(FIXED, 'loto')
euroloto.show_companions(FIXED, 'all', n_top=15)
euroloto.show_companions([13], 'loto', n_top=10)

---
## 11. Prédiction Monte Carlo

`alpha` : `1.0` = fréquence pure · `0.0` = retard pur · `0.6` = défaut.

In [ ]:
euroloto.show_prediction(kind='loto', n=10, alpha=0.6)
euroloto.show_prediction(kind='euro', n=10, alpha=0.5)

In [ ]:
FIXED = [26, 27]
euroloto.show_prediction(FIXED, kind='loto', n=10, alpha=0.6)
euroloto.show_prediction(FIXED, kind='all',  n=5,  alpha=0.6)

---
## 12. Combinaisons optimales

A partir d'une **paire fixée**, calcule les **N combinaisons les plus probables ET les plus inter-entropiques** — aucune paire retenue ne se ressemble (hormis les numéros fixés).

| Colonne | Définition |
|---------|------------|
| **Score lift** | Lift conditionnel moyen sur tous les C(5,2) couples |
| **Diversite** | Distance de Jaccard min au voisin le plus proche dans le set retenu |
| **Prob. (%)** | Probabilité relative parmi toutes les C(NCAND, n_manquants) |

**Algorithme** : filtre tirages contenant FIXED → rank NCAND compagnons → énumère C(NCAND, n_manquants) → score par lift conditionnel → sélectionne N_TOP par greedy Jaccard diversity.

> Rappel : tirages indépendants. Analyse de tendances empiriques.

In [ ]:
FIXED = [26, 46]  # paire fixee
KIND  = 'euro'    # 'loto' | 'euro'
N_TOP = 10        # combinaisons a retenir
NCAND = 35        # pool candidats  (C(35,3) = 6545 combos evaluees)

combos, ref = euroloto.show_combinations(FIXED, KIND, N_TOP, NCAND)

---
## 13. Pipeline complet de prédiction

Enchaîne les 5 étapes en **un seul appel** :

```
FIXED -> Etape 1: Compagnons       -> Etape 2: Analyse profonde (cascade)
      -> Etape 3: Heatmaps affinite -> Etape 4: Combinaisons optimales
      -> Etape 5: Grilles finales (Monte Carlo + bonus)
```

In [ ]:
FIXED = [26, 46]
KIND  = 'euro'

result = euroloto.pipeline(FIXED, KIND, n_top=10, n_candidates=35, k_seeds=3, alpha=0.6)